# Patrón de Comportamiento: Observer (Observador)

## Introducción
El patrón Observer define una dependencia uno-a-muchos entre objetos, de modo que cuando uno cambia de estado, todos sus dependientes son notificados automáticamente.

## Objetivos
- Comprender cómo desacoplar emisores y receptores de eventos.
- Implementar el patrón Observer en Python.
- Comparar la solución con y sin el patrón.

## Ejemplo práctico
Supón que tienes un sistema de noticias donde varios usuarios quieren ser notificados cuando hay una nueva noticia.

### Sin patrón Observer (forma errónea)

In [1]:
class Notificador:
    def __init__(self):
        self.usuarios = []
    def nueva_noticia(self, noticia):
        for usuario in self.usuarios:
            usuario.recibir(noticia)

# Esto acopla Notificador y Usuario fuertemente

### Con patrón Observer (forma correcta)

In [2]:
class Observer:
    def update(self, noticia):
        pass

class Usuario(Observer):
    def __init__(self, nombre):
        self.nombre = nombre
    def update(self, noticia):
        print(f'{self.nombre} recibió: {noticia}')

class Notificador:
    def __init__(self):
        self.observers = []
    def suscribir(self, observer):
        self.observers.append(observer)
    def nueva_noticia(self, noticia):
        for observer in self.observers:
            observer.update(noticia)

notificador = Notificador()
usuario1 = Usuario('Ana')
usuario2 = Usuario('Luis')
notificador.suscribir(usuario1)
notificador.suscribir(usuario2)
notificador.nueva_noticia('¡Nueva noticia!')

Ana recibió: ¡Nueva noticia!
Luis recibió: ¡Nueva noticia!


## UML del patrón Observer
```plantuml
@startuml
interface Observer {
    + update(noticia)
}
class Usuario {
    + update(noticia)
}
Observer <|.. Usuario
class Notificador {
    - observers: list
    + suscribir(observer)
    + nueva_noticia(noticia)
}
Usuario <.. Notificador : notifica
@enduml
```

## Otro ejemplo de la vida real: Sistema de alertas de precio de activos
**Contexto:** una plataforma de trading (acciones, cripto) necesita notificar a distintos consumidores cada vez que el precio de un activo cambia: una alerta por email, un bot de trading automático, y un dashboard en tiempo real. Cada uno reacciona distinto al mismo evento, y el activo no debería tener que conocer los detalles de cada consumidor.

### Sin patrón (forma errónea)
El activo conoce directamente el tipo de suscriptor (una lista de emails) y su lógica de notificación queda fija dentro de la clase.

In [3]:
class Activo:
    def __init__(self, ticker):
        self.ticker = ticker
        self.suscriptores_email = []
    def actualizar_precio(self, precio):
        for email in self.suscriptores_email:
            print(f'Enviando email a {email}: {self.ticker} ahora vale ${precio}')
        # Para agregar un bot de trading o un dashboard, hay que modificar este método

activo = Activo('BTC')
activo.suscriptores_email = ['ana@mail.com']
activo.actualizar_precio(65000)

Enviando email a ana@mail.com: BTC ahora vale $65000


### Con patrón (forma correcta)
Cualquier objeto que implemente `Observador.notificar()` puede suscribirse al activo, sin que `Activo` conozca sus tipos concretos.

In [4]:
class Observador:
    def notificar(self, ticker, precio):
        raise NotImplementedError

class AlertaEmail(Observador):
    def __init__(self, email):
        self.email = email
    def notificar(self, ticker, precio):
        print(f'Email a {self.email}: {ticker} ahora vale ${precio}')

class BotDeTrading(Observador):
    def notificar(self, ticker, precio):
        if precio > 60000:
            print(f'Bot: comprando {ticker} agresivamente a ${precio}')

class DashboardTiempoReal(Observador):
    def notificar(self, ticker, precio):
        print(f'Dashboard actualizado: {ticker} = ${precio}')


class Activo:
    def __init__(self, ticker):
        self.ticker = ticker
        self._observadores = []
    def suscribir(self, observador):
        self._observadores.append(observador)
    def actualizar_precio(self, precio):
        for observador in self._observadores:
            observador.notificar(self.ticker, precio)


btc = Activo('BTC')
btc.suscribir(AlertaEmail('ana@mail.com'))
btc.suscribir(BotDeTrading())
btc.suscribir(DashboardTiempoReal())

btc.actualizar_precio(65000)

Email a ana@mail.com: BTC ahora vale $65000
Bot: comprando BTC agresivamente a $65000
Dashboard actualizado: BTC = $65000


### UML del ejemplo de alertas de precio
```plantuml
@startuml
abstract class Observador {
    + notificar(ticker, precio)
}
class AlertaEmail
class BotDeTrading
class DashboardTiempoReal
Observador <|-- AlertaEmail
Observador <|-- BotDeTrading
Observador <|-- DashboardTiempoReal
class Activo {
    - _observadores: list
    + suscribir(observador)
    + actualizar_precio(precio)
}
Activo --> Observador
@enduml
```

### ¿Dónde más se usa Observer?
- **Alertas de precios/trading:** exactamente este ejemplo — apps como Binance o Robinhood notifican a múltiples canales cuando cambia el precio de un activo.
- **Frameworks de UI reactivos:** React, Vue o Angular re-renderizan componentes "observadores" cuando cambia el estado del que dependen.
- **Sistemas de eventos de dominio (event-driven):** cuando ocurre un evento de negocio (pedido creado, usuario registrado), múltiples handlers se ejecutan sin que el emisor los conozca.
- **Monitoreo e incidentes:** un servicio de métricas que notifica a Slack, PagerDuty y a un dashboard cuando una métrica cruza un umbral.
- **Sistemas de noticias:** el ejemplo con el que abre este notebook — notificar a los usuarios suscritos cuando hay contenido nuevo.

**Ejercicio de reflexión:** si `BotDeTrading` lanzara una excepción al procesar un precio inválido, ¿debería eso impedir que `DashboardTiempoReal` reciba igualmente la notificación? ¿Cómo protegerías el bucle de notificación en `Activo.actualizar_precio()`?

## Actividad
Crea tu propio sistema Observer para notificar cambios en el stock de productos en una tienda.